# Extract Plain Text from `filelist.json_unified`
Queries `filelist` for rows where `nitearticle = 1`, extracts and cleans the `text_plain` field from the `json_unified` column, and saves the result as a CSV with columns `id` and `plain_text`.

In [1]:
import psycopg2
import pandas as pd
import json
import re
from bs4 import BeautifulSoup
from pathlib import Path

In [2]:
# --- Configuration ---
DB_PARAMS = {
    'dbname':   'thecall',
    'user':     'postgres',
    'password': 'password',
    'host':     'localhost',
    'port':     '5432'
}

TABLE_SCHEMA = 'articles'
TABLE_NAME   = 'filelist'

OUTPUT_PATH  = Path(r"D:\development\2026_python\2026_thecall\website7\data\filelist_plaintext.csv")

In [3]:
def extract_plain_text(json_unified):
    """
    Extract and clean plain text from the json_unified field.
    - Parses the JSON
    - Reads text_plain (which contains minimal HTML)
    - Strips HTML tags via BeautifulSoup
    - Drops the first <p> which is the layout name, not article content
    - Collapses excess whitespace
    """
    if json_unified is None:
        return None

    # Parse JSON — handle both string and dict
    if isinstance(json_unified, str):
        try:
            data = json.loads(json_unified)
        except json.JSONDecodeError:
            return None
    else:
        data = json_unified

    text_plain = data.get('text_plain')
    if not text_plain:
        return None

    # Strip HTML tags
    soup = BeautifulSoup(text_plain, 'html.parser')
    paragraphs = soup.find_all('p')

    # Drop first <p> — it is the layout/file name, not article text
    content_paragraphs = paragraphs[1:] if len(paragraphs) > 1 else paragraphs

    text = ' '.join(p.get_text(separator=' ', strip=True) for p in content_paragraphs)

    # Collapse excess whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text if text else None

In [4]:
# --- Query the database ---
conn = psycopg2.connect(**DB_PARAMS)
cur  = conn.cursor()

query = f"""
    SELECT id, json_unified
    FROM {TABLE_SCHEMA}.{TABLE_NAME}
    WHERE nitearticle = 1
"""

cur.execute(query)
rows = cur.fetchall()
print(f"Fetched {len(rows)} rows where nitearticle = 1")

cur.close()
conn.close()
print("Connection closed.")

Fetched 125 rows where nitearticle = 1
Connection closed.


In [5]:
# --- Extract plain text from each row ---
results = []
skipped = 0

for row_id, json_unified in rows:
    plain_text = extract_plain_text(json_unified)
    if plain_text is None:
        skipped += 1
    results.append({'id': row_id, 'plain_text': plain_text})

print(f"Extracted text from {len(results) - skipped} rows")
print(f"Skipped (no text found): {skipped} rows")

Extracted text from 123 rows
Skipped (no text found): 2 rows


In [6]:
# --- Preview first 3 results ---
df = pd.DataFrame(results, columns=['id', 'plain_text'])
df.head(3)

,id,plain_text
0,57603,NaN
1,31789,Monday Nite Footballers Predictions By Jim “Gr...
2,338164,"Monday Niters Chatter, Conversation and Commen..."


In [7]:
# --- Save to CSV ---
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")

Saved 125 rows to D:\development\2026_python\2026_thecall\website7\data\filelist_plaintext.csv
